# CALIBRACIÓN DEL MEDIDOR DE ESPESORES POR ULTRASONIDO

Se calibra el medidor de espesores **INSIZE ISU-100D** (rango $0.8$ a $300$ mm,
resolución $0.01$ mm, transductor de $5$ MHz) por comparación con un patrón
escalonado de acero, siguiendo el procedimiento PC-029 del INACAL y la técnica de
eco de pulso del marco teórico.

Las mediciones se tomaron entre $22.5$ y $23.3\ ^\circ$C y se registran en
`data/ultrasonido.csv`: diez valores patrón y cinco lecturas repetidas por punto.
Este cuaderno reproduce los cálculos de la sección de calibración del informe y
exporta las tablas LaTeX correspondientes.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sympy import symbols, latex, expand, sqrt

In [2]:
def eq(md: str, value, *args):
    disp = f"{md}={value}"
    disp += r"\ \mathrm{ " + " ".join(args) + "}" if args else ""
    display(Markdown(f"$$ {disp}$$"))

## 0. DATOS EXPERIMENTALES Y REDUCCIÓN

El error de indicación elemental de cada punto se define como la diferencia entre
la media de las cinco lecturas y el valor patrón del escalón:

$$E_k = \left(\bar d_k - VP_k\right)\times 10^3\ \mu\text{m}.$$

La repetibilidad se estima con la desviación estándar muestral insesgada y la
incertidumbre de la media, ambas deducidas en el capítulo 1:

$$\sigma(\bar d_k)=\frac{s_k}{\sqrt{n}},\qquad
s_k^2=\frac{1}{n-1}\sum_{p=1}^{n}\left(d_{kp}-\bar d_k\right)^2,$$

con $n=5$ lecturas por punto.

In [3]:
datos = pd.read_csv("data/ultrasonido.csv")
datos

,N,VP_mm,L1_mm,L2_mm,L3_mm,L4_mm,L5_mm
0,1,2.5,2.51,2.50,2.51,2.51,2.50
1,2,5.0,5.05,4.98,5.00,4.98,4.98
2,3,7.5,7.47,7.54,7.53,7.51,7.49
3,4,10.0,10.00,9.98,10.01,10.01,10.01
4,5,12.5,12.54,12.52,12.51,12.51,12.50
5,6,15.0,15.01,15.00,14.99,14.96,14.99
6,7,17.5,17.52,17.55,17.47,17.46,17.49
7,8,20.0,20.01,19.99,19.98,19.97,20.02
8,9,22.5,22.52,22.54,22.52,22.48,22.53
9,10,25.0,25.05,25.03,25.00,24.99,24.97


In [4]:
from pathlib import Path

TAB_DIR = Path("inf/c3-calibration/ultrasonido/tablas")
TAB_DIR.mkdir(parents=True, exist_ok=True)


def fmt(v, dec=0):
    v = 0.0 if abs(v) < 0.5 * 10 ** (-dec) else v
    return f"{v:.{dec}f}"


def export_table(path, header, rows, caption, label, colspec):
    body = [
        r"\begin{table}[H]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
        " & ".join(header) + r" \\",
        r"\midrule",
    ]
    body += [" & ".join(row) + r" \\" for row in rows]
    body += [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""]
    Path(path).write_text("\n".join(body), encoding="utf-8")
    print(f"escrito: {path}")

In [5]:
n = 5

VP = datos["VP_mm"].to_numpy()
L = datos[[f"L{i}_mm" for i in range(1, n + 1)]].to_numpy()

d_media = L.mean(axis=1)                 # mm
s_L = L.std(axis=1, ddof=1)              # mm
u_rep = s_L / np.sqrt(n) * 1e3           # um
E = (d_media - VP) * 1e3                 # um

resumen = pd.DataFrame(
    {"VP (mm)": VP, "media (mm)": d_media, "E (um)": E,
     "s (um)": s_L * 1e3, "u_rep (um)": u_rep}
)
display(resumen)

eq(r"E_{\max}", f"{E.max():.0f}", r"\mu m")
eq(r"E_{\min}", f"{E.min():.0f}", r"\mu m")
eq(r"f_e = E_{\max} - E_{\min}", f"{E.max() - E.min():.0f}", r"\mu m")
eq(r"\sigma(\bar d)_{\max}", f"{u_rep.max():.2f}", r"\mu m")

,VP (mm),media (mm),E (um),s (um),u_rep (um)
0,2.5,2.506,6.0,5.477226,2.449490
1,5.0,4.998,-2.0,30.331502,13.564660
2,7.5,7.508,8.0,28.635642,12.806248
3,10.0,10.002,2.0,13.038405,5.830952
4,12.5,12.516,16.0,15.165751,6.782330
5,15.0,14.990,-10.0,18.708287,8.366600
6,17.5,17.498,-2.0,37.013511,16.552945
7,20.0,19.994,-6.0,20.736441,9.273618
8,22.5,22.518,18.0,22.803509,10.198039
9,25.0,25.008,8.0,31.937439,14.282857


$$ E_{\max}=18\ \mathrm{ \mu m}$$

$$ E_{\min}=-10\ \mathrm{ \mu m}$$

$$ f_e = E_{\max} - E_{\min}=28\ \mathrm{ \mu m}$$

$$ \sigma(\bar d)_{\max}=16.55\ \mathrm{ \mu m}$$

In [6]:
export_table(
    TAB_DIR / "t1.tex",
    header=[
        r"$N$",
        r"$VP$ (mm)",
        r"$d_1$ (mm)",
        r"$d_2$ (mm)",
        r"$d_3$ (mm)",
        r"$d_4$ (mm)",
        r"$d_5$ (mm)",
    ],
    rows=[
        [str(k), fmt(vp, 2)] + [fmt(v, 2) for v in fila]
        for k, (vp, fila) in enumerate(zip(VP, L), start=1)
    ],
    caption=(
        "Mediciones de calibración del medidor de espesores por ultrasonido: cinco "
        "lecturas repetidas por valor patrón del escalón."
    ),
    label="tab:ut-datos",
    colspec="rrrrrrr",
)

escrito: inf/c3-calibration/ultrasonido/tablas/t1.tex


In [7]:
export_table(
    TAB_DIR / "t2.tex",
    header=[
        r"$VP$ (mm)",
        r"$\bar d$ (mm)",
        r"$s$ ($\mu$m)",
        r"$s/\sqrt{n}$ ($\mu$m)",
        r"$E$ ($\mu$m)",
    ],
    rows=[
        [fmt(vp, 2), fmt(m, 3), fmt(s, 2), fmt(sm, 2), fmt(e, 0)]
        for vp, m, s, sm, e in zip(VP, d_media, s_L * 1e3, u_rep, E)
    ],
    caption=(
        "Cálculo estadístico elemental por punto de calibración. $\\bar d$ es la "
        "media de las cinco lecturas, $s$ la desviación estándar muestral y $E$ el "
        "error de indicación sin correcciones."
    ),
    label="tab:ut-estadistica",
    colspec="rrrrr",
)

escrito: inf/c3-calibration/ultrasonido/tablas/t2.tex


### 0.1 INTERPRETACIÓN DE LA DISPERSIÓN

La Tabla~\ref{tab:ut-datos} recoge las lecturas y la
Tabla~\ref{tab:ut-estadistica} su reducción. El error de indicación máximo es
$E=18\ \mu$m en $22.50$ mm y el mínimo $-10\ \mu$m en $15.00$ mm, de modo que el
alcance del error es $f_e=28\ \mu$m.

La repetibilidad no es uniforme a lo largo del alcance: el punto de $17.50$ mm
concentra la mayor dispersión de la media, $\sigma(\bar d)=16.55\ \mu$m, mientras
que en $2.50$ mm apenas llega a $2.45\ \mu$m. Esa variación se traslada a la
incertidumbre combinada punto a punto.

## 1. COMPONENTES DE INCERTIDUMBRE

Las componentes tipo B se acotan con la distribución rectangular del capítulo 1,
$\sigma=a/\sqrt{3}$, o con el certificado del patrón cuando existe.

### 1.1 Repetibilidad

Única componente tipo A. Se toma la desviación estándar de la media de las cinco
lecturas en cada punto, calculada en la sección 0.

### 1.2 Resolución del instrumento

La pantalla del ISU-100D resuelve $R=0.01$ mm, de modo que

$$\sigma(\delta l_{res})=\frac{R}{2\sqrt{3}}=\frac{0.01}{\sqrt{12}}\ \text{mm}
\approx 2.89\ \mu\text{m}.$$

### 1.3 Patrón de espesor

La incertidumbre del patrón escalonado combina el certificado de calibración, con
$U_{cert}=0.05$ mm y $k=2$, y la deriva dimensional máxima $U_{der}=0.1$ mm
tratada como rectangular:

$$\sigma^2(V_{PE})=\left(\frac{U_{cert}}{k}\right)^2
+\frac{U_{der}^2}{3}.$$

### 1.4 Coeficiente de dilatación

El coeficiente del acero es $\alpha=11.5\times10^{-6}\ ^\circ$C$^{-1}$ con una
tolerancia del $10\ \%$, acotada de forma rectangular:

$$\sigma(\alpha)=\frac{0.1\,\alpha}{\sqrt{3}}
\approx 6.7\times10^{-7}\ ^\circ\text{C}^{-1}.$$

### 1.5 Variación de temperatura

Con la exactitud del termómetro $U_{Tcert}=0.5\ ^\circ$C ($k=2$) y la fluctuación
observada $\Delta t_A=0.8\ ^\circ$C:

$$\sigma(\Delta T_i)=\sqrt{\left(\frac{U_{Tcert}}{k}\right)^2
+\left(\frac{\Delta t_A}{\sqrt{3}}\right)^2}.$$

In [8]:
R_res = 0.01                   # mm, resolucion del display
U_cert_PE, k_PE = 0.05, 2      # mm, certificado del patron escalonado
U_der_PE = 0.1                 # mm, deriva maxima
alpha_PE = 11.5e-6             # 1/C, acero
UT_cert, kT = 0.5, 2           # C, termometro de contacto
t_ini, t_fin = 22.5, 23.3      # C, del acta de calibracion

u_res = R_res / np.sqrt(12) * 1e3                                    # um
u_PE = np.sqrt((U_cert_PE / k_PE) ** 2
               + (U_der_PE / np.sqrt(3)) ** 2) * 1e3                 # um
u_alpha = 0.1 * alpha_PE / np.sqrt(3)                                # 1/C
u_DT = np.sqrt((UT_cert / kT) ** 2
               + ((t_fin - t_ini) / np.sqrt(3)) ** 2)                # C

DT_val = (t_ini + t_fin) / 2 - 20.0                                  # C
c3 = 1 + alpha_PE * DT_val

eq(r"\sigma(\delta l_{res})", f"{u_res:.2f}", r"\mu m")
eq(r"\sigma(V_{PE})", f"{u_PE:.2f}", r"\mu m")
eq(r"\sigma(\alpha)", f"{u_alpha:.3g}", r"^\circ C^{-1}")
eq(r"\sigma(\Delta T_i)", f"{u_DT:.4g}", r"^\circ C")
eq(r"c_3 = 1 + \alpha\,\Delta T_i", f"{c3:.7f}")

$$ \sigma(\delta l_{res})=2.89\ \mathrm{ \mu m}$$

$$ \sigma(V_{PE})=62.92\ \mathrm{ \mu m}$$

$$ \sigma(\alpha)=6.64e-07\ \mathrm{ ^\circ C^{-1}}$$

$$ \sigma(\Delta T_i)=0.5252\ \mathrm{ ^\circ C}$$

$$ c_3 = 1 + \alpha\,\Delta T_i=1.0000334$$

## 2. MODELO DE MEDICIÓN Y LEY DE PROPAGACIÓN

El espesor indicado se obtiene del tiempo de vuelo, $d=c_L\Delta t/2$. Al
compararlo con el patrón y corregir la dilatación térmica respecto de
$T_0=20\ ^\circ$C, el error de indicación es

$$E = \bar d\,(1 - \alpha_{inst}\,\Delta T_{inst})
- V_{PE}\,(1 - \alpha_{PE}\,\Delta T_{PE}).$$

Con $\bar d \approx V_{PE}$ en los términos correctivos y un único coeficiente
$\alpha$ para el acero, el modelo linealizado queda

$$E = f(\bar d;\,\delta l_{res};\,V_{PE};\,\alpha;\,\Delta T_i)
= \bar d - V_{PE} - V_{PE}\,\alpha\,\Delta T_i .$$

Según la ley de propagación del capítulo 1, los coeficientes de sensibilidad son

$$c_1=\frac{\partial E}{\partial \bar d}=1,\qquad
c_2=\frac{\partial E}{\partial (\delta l_{res})}=1,\qquad
c_3=\frac{\partial E}{\partial V_{PE}}=-(1+\alpha\,\Delta T_i),$$

$$c_4=\frac{\partial E}{\partial \alpha}=-V_{PE}\,\Delta T_i,\qquad
c_5=\frac{\partial E}{\partial \Delta T_i}=-V_{PE}\,\alpha,$$

de modo que la varianza combinada del error de indicación es

$$\sigma_c^2(E)=\sigma^2(\bar d)+\sigma^2(\delta l_{res})
+\left[1+\alpha\,\Delta T_i\right]^2\sigma^2(V_{PE})
+\left[V_{PE}\,\Delta T_i\right]^2\sigma^2(\alpha)
+\left[V_{PE}\,\alpha\right]^2\sigma^2(\Delta T_i).$$

Con $k=2$, la incertidumbre expandida es $U(E)=2\,\sigma_c(E)$.

In [9]:
L_sym = symbols("L")           # L en um

u_rep_max = u_rep.max()
u2c_const = u_rep_max ** 2 + u_res ** 2 + c3 ** 2 * u_PE ** 2
u2c_expr = u2c_const + (L_sym * DT_val) ** 2 * u_alpha ** 2 \
    + (L_sym * alpha_PE) ** 2 * u_DT ** 2

u2cE = expand(u2c_expr)
eq(r"\sigma_c^2(E,L)", latex(u2cE), r"\mu m^2")
eq(r"\sigma_c(E,L)", latex(sqrt(u2cE)), r"\mu m")
eq(r"U(E,L)", latex(sqrt(4 * u2cE)), r"\mu m")

$$ \sigma_c^2(E,L)=4.01863666666667 \cdot 10^{-11} L^{2} + 4240.93069190255\ \mathrm{ \mu m^2}$$

$$ \sigma_c(E,L)=\sqrt{4.01863666666667 \cdot 10^{-11} L^{2} + 4240.93069190255}\ \mathrm{ \mu m}$$

$$ U(E,L)=\sqrt{1.60745466666667 \cdot 10^{-10} L^{2} + 16963.7227676102}\ \mathrm{ \mu m}$$

## 3. RESULTADOS PUNTO A PUNTO

Cada punto tiene su propia repetibilidad, $\sigma(\bar d_k)=s_k/\sqrt5$. Las
componentes restantes son comunes al alcance, salvo los dos términos térmicos, que
crecen con el espesor. La Tabla~\ref{tab:ut-componentes} recoge el conjunto y el
resultado de cada punto.

In [10]:
t_rep = u_rep
t_res = np.full_like(VP, u_res)
t_PE = np.full_like(VP, c3 * u_PE)
t_alpha = (VP * 1e3 * DT_val) * u_alpha
t_DT = (VP * 1e3 * alpha_PE) * u_DT

u_c = np.sqrt(t_rep ** 2 + t_res ** 2 + t_PE ** 2 + t_alpha ** 2 + t_DT ** 2)
U = 2 * u_c

tabla = pd.DataFrame(
    {"VP (mm)": VP, "E (um)": E, "s(d) (um)": t_rep, "c3 s(VPE) (um)": t_PE,
     "s(res) (um)": t_res, "s(alpha) (um)": t_alpha, "s(dT) (um)": t_DT,
     "s_c (um)": u_c, "U (um)": U}
)
display(tabla)

eq(r"U_{\min}", f"{U.min():.1f}", r"\mu m")
eq(r"U_{\max}", f"{U.max():.1f}", r"\mu m")

export_table(
    TAB_DIR / "t3.tex",
    header=[
        r"$VP$ (mm)",
        r"$\sigma(\bar d)$ ($\mu$m)",
        r"$c_3\,\sigma(V_{PE})$ ($\mu$m)",
        r"$\sigma(\delta l_{res})$ ($\mu$m)",
        r"$\sigma_c(E)$ ($\mu$m)",
        r"$U(E)$ ($\mu$m)",
    ],
    rows=[
        [fmt(vp, 2), fmt(tr, 2), fmt(tpe, 2), fmt(tre, 2), fmt(uc, 2), fmt(u, 1)]
        for vp, tr, tpe, tre, uc, u in zip(VP, t_rep, t_PE, t_res, u_c, U)
    ],
    caption=(
        "Componentes de incertidumbre típica y resultados de la calibración punto "
        "a punto. Los términos térmicos se detallan en la "
        "Tabla~\\ref{tab:ut-balance}."
    ),
    label="tab:ut-componentes",
    colspec="rrrrrr",
)

,VP (mm),E (um),s(d) (um),c3 s(VPE) (um),s(res) (um),s(alpha) (um),s(dT) (um),s_c (um),U (um)
0,2.5,6.0,2.449490,62.917385,2.886751,0.004814,0.015099,63.031190,126.062380
1,5.0,-2.0,13.564660,62.917385,2.886751,0.009627,0.030199,64.427725,128.855449
2,7.5,8.0,12.806248,62.917385,2.886751,0.014441,0.045298,64.272334,128.544669
3,10.0,2.0,5.830952,62.917385,2.886751,0.019255,0.060398,63.252942,126.505885
4,12.5,16.0,6.782330,62.917385,2.886751,0.024068,0.075497,63.347746,126.695493
5,15.0,-10.0,8.366600,62.917385,2.886751,0.028882,0.090597,63.536916,127.073833
6,17.5,-2.0,16.552945,62.917385,2.886751,0.033696,0.105696,65.122523,130.245046
7,20.0,-6.0,9.273618,62.917385,2.886751,0.038509,0.120796,63.662758,127.325516
8,22.5,18.0,10.198039,62.917385,2.886751,0.043323,0.135895,63.804005,127.608010
9,25.0,8.0,14.282857,62.917385,2.886751,0.048137,0.150995,64.582937,129.165875


$$ U_{\min}=126.1\ \mathrm{ \mu m}$$

$$ U_{\max}=130.2\ \mathrm{ \mu m}$$

escrito: inf/c3-calibration/ultrasonido/tablas/t3.tex


In [11]:
i = int(np.argmax(u_rep))
comp = [
    (r"$\sigma(\bar d)$ (repetibilidad)", t_rep[i]),
    (r"$\sigma(\delta l_{res})$ (resolución)", t_res[i]),
    (r"$c_3\,\sigma(V_{PE})$ (patrón)", t_PE[i]),
    (r"$(V_{PE}\Delta T_i)\,\sigma(\alpha)$", t_alpha[i]),
    (r"$(V_{PE}\alpha)\,\sigma(\Delta T_i)$", t_DT[i]),
]
u_c_i = u_c[i]
var = np.array([v ** 2 for _, v in comp])
tot = var.sum()

for nombre, valor in comp:
    print(f"{nombre:44s} {valor:8.4f} um  var {valor**2:10.4f} um2  ({100*valor**2/tot:6.2f} %)")
print(f"{'sigma_c(E)':44s} {u_c_i:8.2f} um  var {tot:10.2f} um2")
print(f"{'U(E) = 2 sigma_c(E)':44s} {2*u_c_i:8.2f} um")

rows = []
for (nombre, valor), v in zip(comp, var):
    peso = 100 * v / tot
    rows.append([nombre, f"{valor:.3f}", f"{v:.4f}",
                 f"{peso:.2f}" if peso >= 0.005 else r"$<0.01$"])
rows += [
    [r"$\sigma_c(E)$", f"{u_c_i:.2f}", f"{tot:.2f}", "100.00"],
    [r"$U(E) = 2\,\sigma_c(E)$", f"{2*u_c_i:.2f}", r"—", r"—"],
]

export_table(
    TAB_DIR / "t4.tex",
    header=[
        r"Contribución",
        r"$\sigma_i$ ($\mu$m)",
        r"$\sigma_i^2$ ($\mu$m$^2$)",
        r"Peso en la varianza (\%)",
    ],
    rows=rows,
    caption=(
        "Balance de incertidumbre del error de indicación en el punto de mayor "
        "repetibilidad (17.50 mm)."
    ),
    label="tab:ut-balance",
    colspec="lrrr",
)

$\sigma(\bar d)$ (repetibilidad)              16.5529 um  var   274.0000 um2  (  6.46 %)
$\sigma(\delta l_{res})$ (resolución)          2.8868 um  var     8.3333 um2  (  0.20 %)
$c_3\,\sigma(V_{PE})$ (patrón)                62.9174 um  var  3958.5974 um2  ( 93.34 %)
$(V_{PE}\Delta T_i)\,\sigma(\alpha)$           0.0337 um  var     0.0011 um2  (  0.00 %)
$(V_{PE}\alpha)\,\sigma(\Delta T_i)$           0.1057 um  var     0.0112 um2  (  0.00 %)
sigma_c(E)                                      65.12 um  var    4240.94 um2
U(E) = 2 sigma_c(E)                            130.25 um
escrito: inf/c3-calibration/ultrasonido/tablas/t4.tex


La incertidumbre expandida del medidor de espesores crece de $126.1\ \mu$m a
$130.2\ \mu$m en el intervalo ensayado. El $93.3\ \%$ de la varianza combinada
proviene de la deriva del patrón escalonado ($62.9\ \mu$m), seguida de la
repetibilidad ($16.6\ \mu$m en el peor punto) y de la resolución
($2.89\ \mu$m). Los dos términos térmicos suman $0.012\ \mu\text{m}^2$, es decir,
menos del $0.001\ \%$ de la varianza total.

Ese reparto explica por qué el resultado está dominado por el patrón y no por el
instrumento: con un patrón de menor deriva, la incertidumbre del ISU-100D bajaría
casi al nivel de su repetibilidad. El crecimiento con el espesor es despreciable
frente a la variación punto a punto de la repetibilidad.